# Faruq-v3 — AF2 same-device efficiency audit

Mengukur D0FT vs AF2 pada seed 42/123/2026 dalam pasangan pada GPU/runtime yang sama. Tidak ada training, dataset tidak dipulihkan, dan test tidak diakses. Output pasangan tersimpan di Drive dan dapat dipakai kembali bila SHA checkpoint identik.

In [ ]:
BRANCH = 'agent/af2-continuation-confirmation'
DEVICE = '0'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name == 'coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('GPU:',torch.cuda.get_device_name(0),'| BRANCH:',BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
D0FT_REL = [
 'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt',
]
AF2_REL = [
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
 'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt',
 'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt',
]
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=tuple(D0FT_REL + AF2_REL))
D0FT = [require_project_artifact(PROJECT_ROOT,p) for p in D0FT_REL]
AF2 = [require_project_artifact(PROJECT_ROOT,p) for p in AF2_REL]
OUTPUT = PROJECT_ROOT/'experiments/faruq-v3-af2-efficiency-audit-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT_ROOT); print('OUTPUT:',OUTPUT)

In [ ]:
command = [sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_efficiency_audit',
 '--d0ft-checkpoints',*[str(p) for p in D0FT],
 '--af2-checkpoints',*[str(p) for p in AF2],
 '--output-root',str(OUTPUT),'--device',DEVICE]
LOG = OUTPUT/'efficiency_audit_run.log'
print('MENJALANKAN AUDIT EFISIENSI | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process = subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
seen = None
while process.poll() is None:
    ready = len(list((OUTPUT/'pair_reports').glob('D0FT_vs_AF2_seed*.json'))) if (OUTPUT/'pair_reports').is_dir() else 0
    if ready != seen:
        print(f'Pasangan selesai: {ready}/3',flush=True); seen = ready
    time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:]))
    raise RuntimeError(f'Audit efisiensi gagal: {process.returncode}')
SUMMARY = OUTPUT/'af2_efficiency_summary.json'
assert SUMMARY.is_file(),SUMMARY
print('SELESAI:',SUMMARY)

In [ ]:
import pandas as pd
from IPython.display import display
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
rows = []
for metric in ['parameter_count','state_tensor_bytes','checkpoint_file_bytes','latency_median_ms','latency_p95_ms','throughput_images_per_second','peak_allocated_bytes','incremental_inference_peak_bytes']:
    row = summary['aggregate'][metric]
    rows.append({'metric':metric,'D0FT':row['d0ft_mean'],'AF2':row['af2_mean'],'delta':row['delta_mean'],'AF2/D0FT':row['ratio_mean']})
display(pd.DataFrame(rows))
print('GATES:',summary['gates'])
print('PARAMETER-FREE FRONTEND:',summary['parameter_free_frontend_supported'])
print('TRAINING:',summary['training_executed'],'| TEST:',summary['test_images_accessed'])
print('SCOPE:',summary['settings']['timing_scope'])
print('Kirim tabel, gates, dan path summary. Tidak ada training atau test.')